# 12 - Feature attribution

RQ3 asks whether any benefit from fusion depends on how consistently the supervised detector uses its features across generator families. §3.5 measures that as the stability of each family's feature importances against the other six.

The 14 features reach the output through one linear layer with no activation, so each feature's Shapley value has a closed form (Lundberg and Lee, 2017, Corollary 1, Linear SHAP):

    phi_j(x) = w_j (x_j - E[x_j])

The scaler is fitted on each run's training rows, so the standardised training mean is zero and this is simply w_j x_j.

Only the seed-0 model weights were saved, so the analysis uses the seed-0 model of each condition.

1. Do the 14 features move the output at all?
2. Which features does the detector rely on for each held-out family?
3. How differently does it read each family from the other six?
4. Does fusion help most where that difference is largest? (RQ3, H3)

## 1. Load the models

In [1]:
import numpy as np
import pandas as pd
import torch

from logos.paths import DATA_DIR
from logos.features.stylometric import FEATURE_NAMES
from logos.project_data.mage import families
from scipy.stats import spearmanr

MAGE_DIR = DATA_DIR / "mage"
MODELS_DIR = MAGE_DIR / "models" / "stylometric_neural"

# The classifier takes 782 inputs: DistilBERT's 768-dim [CLS] output followed
# by the 14 standardised features, so the feature weights are the last 14
weights, scalers = {}, {}

for family in families:
    name = f"seed0_LOFO_{family.replace(' ', '_')}.pt"
    saved = torch.load(MODELS_DIR / name, map_location="cpu", weights_only=False)
    weights[family] = (
        saved["state_dict"]["classifier.weight"].squeeze(0)[-len(FEATURE_NAMES):].numpy()
    )
    scalers[family] = saved["scaler"]

print(f"loaded {len(weights)} models")

loaded 7 models


## 2. Do the features change any decisions?

Notebook 09 found that no feature separates the classes on MAGE in aggregate, which leaves open whether the 14 contribute anything inside the detector.

The logit is one linear layer over 782 inputs, so it splits exactly into the part coming from DistilBERT's 768 and the part coming from the 14 features. Removing the second and counting how many texts cross the threshold answers the question directly: if none do, the features are decoration.

This is the only cell in the notebook that runs the encoder. A sample is enough to answer it; raise SAMPLE to use more.

By SHAP's local accuracy property (Lundberg and Lee, 2017, Property 1), the 14 features' Shapley values add up to exactly this feature part. Removing it is the same as setting every feature to its training mean.

In [2]:
from transformers import DistilBertTokenizerFast

from logos.detectors.stylometric_neural import StylometricDetector
from logos.evaluation.metrics import calibrate_threshold

SAMPLE = 5_000
BATCH = 64

features = pd.read_parquet(
    MAGE_DIR / "features" / "stylometric.parquet"
).set_index("text_id")

meta = pd.read_parquet(
    MAGE_DIR / "corpus.parquet", columns=["text_id", "label", "family", "split"]
)
test = meta[meta["split"] == "test"]

# The threshold this model actually decides at, from its own saved scores
run = pd.read_parquet(
    MAGE_DIR / "scores" / "stylometric_neural" / "seed0_In-distribution.parquet"
)
calibration = run[run["part"] == "calibration"].merge(meta, on="text_id")
threshold = calibrate_threshold(
    calibration["score"].to_numpy(), calibration["label"].to_numpy()
)

saved = torch.load(
    MODELS_DIR / "seed0_In-distribution.pt", map_location="cpu", weights_only=False
)
classifier = saved["state_dict"]["classifier.weight"].squeeze(0).numpy()
text_weights, feature_weights = classifier[:-len(FEATURE_NAMES)], classifier[-len(FEATURE_NAMES):]
bias = float(saved["state_dict"]["classifier.bias"])

device = "cuda" if torch.cuda.is_available() else "cpu"
model = StylometricDetector()
model.load_state_dict(saved["state_dict"])
model.eval().to(device)

texts = pd.read_parquet(
    MAGE_DIR / "corpus.parquet", columns=["text_id", "text"]
).set_index("text_id")

sample = test.sample(SAMPLE, random_state=0)
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
tokens = tokenizer(
    texts.loc[sample["text_id"], "text"].tolist(),
    padding="max_length", truncation=True, max_length=256, return_tensors="pt",
)

cls = []
with torch.no_grad():
    for start in range(0, len(sample), BATCH):
        hidden = model.distilbert(
            input_ids=tokens["input_ids"][start:start + BATCH].to(device),
            attention_mask=tokens["attention_mask"][start:start + BATCH].to(device),
        ).last_hidden_state[:, 0]
        cls.append(hidden.cpu().numpy())

cls = np.concatenate(cls)
standardised = saved["scaler"].transform(features.loc[sample["text_id"], FEATURE_NAMES])

text_part = cls @ text_weights + bias
feature_part = standardised @ feature_weights

flipped = (text_part >= threshold) != (text_part + feature_part >= threshold)

print(f"spread of the text part     {text_part.std():.3f}")
print(f"spread of the feature part  {feature_part.std():.3f}")
print(f"threshold                   {threshold:.3f}")
print(f"\ndecisions changed by the 14 features: {flipped.sum():,} of {len(sample):,} "
      f"({flipped.mean():.2%})")

W1120 18:34:26.791000 28199 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


spread of the text part     6.679
spread of the feature part  0.087
threshold                   7.616

decisions changed by the 14 features: 36 of 5,000 (0.72%)


/Users/xwm517/anaconda3/envs/logos/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


## 3. Feature importance per family

A family's importance vector is the mean absolute Shapley value of each feature (Lundberg et al., 2020), across that family's machine test texts, under the model trained without that family. Comparing global SHAP importance across datasets follows Pudasaini et al. (2026). Only machine texts carry a family label, so human texts are not included.

In [3]:
machine_test = test[test["label"] == 1]

importance = {}
for family in families:
    ids = machine_test.loc[machine_test["family"] == family, "text_id"]
    standardised = scalers[family].transform(features.loc[ids, FEATURE_NAMES])
    importance[family] = np.abs(standardised * weights[family]).mean(axis=0)

importance = pd.DataFrame(importance, index=FEATURE_NAMES)
importance.round(3)

/Users/xwm517/anaconda3/envs/logos/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/Users/xwm517/anaconda3/envs/logos/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/Users/xwm517/anaconda3/envs/logos/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/Users/xwm517/anaconda3/envs/logos/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
/Users/xwm517/anaconda3/envs/logos/lib/python3.11/site-packages/sklearn/utils/validation.py:2823: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.w

,OpenAI GPT,LLaMA,GLM-130B,FLAN-T5,OPT,BigScience,EleutherAI
sentence_length_mean,0.009,0.012,0.010,0.020,0.013,0.011,0.015
sentence_length_sd,0.003,0.003,0.003,0.006,0.007,0.004,0.008
mattr,0.006,0.017,0.019,0.015,0.011,0.011,0.014
hapax_rate,0.017,0.045,0.059,0.035,0.026,0.027,0.030
stopword_rate,0.012,0.015,0.018,0.017,0.016,0.015,0.017
punctuation_rate,0.012,0.015,0.018,0.015,0.011,0.014,0.011
noun_rate,0.016,0.020,0.020,0.021,0.017,0.019,0.016
verb_rate,0.016,0.019,0.020,0.022,0.013,0.018,0.011
past_tense_rate,0.031,0.029,0.032,0.036,0.022,0.029,0.021
adverb_rate,0.016,0.014,0.015,0.021,0.015,0.018,0.016


## 4. How differently is each family read?

A family's instability is one minus the mean Spearman correlation between its importance vector and those of the other six (§3.5). A high value means the detector relies on a different set of features for that family than it does elsewhere.

Kalousis et al. (2007) measure the stability of feature rankings in the same way, as the average Spearman correlation between rankings.

In [4]:
correlations = importance.corr(method="spearman")
instability = 1 - (correlations.sum() - 1) / (len(correlations) - 1)
instability = instability.sort_values(ascending=False).rename("instability")

print(instability.round(3).to_string())
correlations.round(3)

EleutherAI    0.253
GLM-130B      0.168
LLaMA         0.155
FLAN-T5       0.136
OpenAI GPT    0.128
OPT           0.118
BigScience    0.100


,OpenAI GPT,LLaMA,GLM-130B,FLAN-T5,OPT,BigScience,EleutherAI
OpenAI GPT,1.000,0.846,0.833,0.934,0.899,0.982,0.736
LLaMA,0.846,1.000,0.987,0.842,0.851,0.881,0.662
GLM-130B,0.833,0.987,1.000,0.824,0.824,0.868,0.657
FLAN-T5,0.934,0.842,0.824,1.000,0.886,0.947,0.749
OPT,0.899,0.851,0.824,0.886,1.000,0.938,0.895
BigScience,0.982,0.881,0.868,0.947,0.938,1.000,0.780
EleutherAI,0.736,0.662,0.657,0.749,0.895,0.780,1.000


## 5. Against the fusion gains

H3 predicts fusion helps most where instability is highest. This is tested with Spearman's rank correlation between instability and fusion gain across the seven families (Spearman, 1904). With seven points, only a strong relationship could reach significance.

In [5]:
fusion = pd.read_csv(DATA_DIR / "results" / "summary_fusion.csv")
stylometric = pd.read_csv(DATA_DIR / "results" / "summary_stylometric.csv")


def lofo(frame):
    return frame[frame["condition"] == "LOFO"].set_index("family")["tpr_mean"]


h3 = pd.DataFrame({
    "instability": instability,
    "stylometric_neural": lofo(stylometric),
    "fusion": lofo(fusion),
})
h3["fusion_gain"] = h3["fusion"] - h3["stylometric_neural"]

rho, p = spearmanr(h3["instability"], h3["fusion_gain"])
print(f"Spearman rho = {rho:.2f}, p = {p:.2f}")

h3.sort_values("instability", ascending=False).round(4)

Spearman rho = 0.14, p = 0.76


,instability,stylometric_neural,fusion,fusion_gain
EleutherAI,0.2535,0.9402,0.8630,-0.0772
GLM-130B,0.1678,0.8065,0.8472,0.0407
LLaMA,0.1553,0.6625,0.6486,-0.0139
FLAN-T5,0.1363,0.1118,0.1360,0.0242
OpenAI GPT,0.1282,0.3576,0.3802,0.0226
OPT,0.1179,0.6442,0.4535,-0.1907
BigScience,0.1004,0.4562,0.4687,0.0125


## 6. Save

In [6]:
out_path = DATA_DIR / "results" / "feature_importance.csv"
importance.round(4).to_csv(out_path)

print(f"saved importance vectors to {out_path}")
h3.round(4).to_csv(DATA_DIR / "results" / "instability.csv")
print("saved instability and fusion gains")

saved importance vectors to /Users/xwm517/Desktop/logos/data/results/feature_importance.csv
saved instability and fusion gains
